# Accessing Claude with the API: Controlling Output

This notebook covers:
1. **Structured Data**: Generating strict structures (like JSON) by using prefilled assistant messages.
2. **Stop Sequences**: Halting output generation immediately when a specified token (like closing markdown fences) is generated.
3. **Structured Data Exercise**: Parsing generated strings into native Python dictionaries using `json.loads`.

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helpers to write history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 600,
        "messages": messages
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.create(**params)
    return message.content[0].text

## 1. Prefilled Responses & Stop Sequences

In [4]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

# Prefill the response with "```json" to force Claude to write only JSON and not intro text
add_assistant_message(messages, "```json")

# Set "```" as a stop sequence so that Claude stops generating immediately upon completing the JSON
text = chat(messages, stop_sequences=["```"])

print("=== Raw Response ===")
print(text)

=== Raw Response ===

{
  "Name": "MyRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



## 2. Structured Data Exercise (Parsing JSON)

In [5]:
import json

try:
    # Clean and parse the response into a python dictionary
    json_data = json.loads(text.strip())
    print("=== Successfully Parsed Dictionary ===")
    print(type(json_data))
    print(json_data)
except json.JSONDecodeError as e:
    print("JSON parsing failed! Check raw text.", e)

=== Successfully Parsed Dictionary ===
<class 'dict'>
{'Name': 'MyRule', 'EventBusName': 'default', 'EventPattern': {'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running']}}, 'State': 'ENABLED', 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction', 'Id': '1'}]}
